# Tokenization: From Scratch

Deep Learning models cannot process raw text directly. They require numbers. **Tokenization** is the process of converting text into a sequence of numbers (tokens).

In this tutorial, we will:
1. Build a simple **Character-level Tokenizer** from scratch.
2. Understand **Byte Pair Encoding (BPE)**, the algorithm used by modern LLMs.
3. Use the production-ready `AutoTokenizer` from Hugging Face.

## 1. Character-Level Tokenizer (From Scratch)

The simplest way to tokenize is to map every character to a unique integer.

In [1]:
text = "Hello, World! This is a test."

# 1. Create the vocabulary (all unique characters)
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary: {chars}")
print(f"Vocab Size: {vocab_size}")

# 2. Create mappings
stoi = { ch:i for i,ch in enumerate(chars) } # String to Integer
itos = { i:ch for i,ch in enumerate(chars) } # Integer to String

# 3. Encoder: Text -> Integers
encode = lambda s: [stoi[c] for c in s]

# 4. Decoder: Integers -> Text
decode = lambda l: ''.join([itos[i] for i in l])

# Test it
encoded = encode("Hello")
decoded = decode(encoded)

print(f"Encoded 'Hello': {encoded}")
print(f"Decoded: {decoded}")

Vocabulary: [' ', '!', ',', '.', 'H', 'T', 'W', 'a', 'd', 'e', 'h', 'i', 'l', 'o', 'r', 's', 't']
Vocab Size: 17
Encoded 'Hello': [4, 9, 12, 12, 13]
Decoded: Hello


### Limitations
Character-level models have a very small vocabulary but result in very long sequences. The model has to learn how to spell words from individual letters.

## 2. Byte Pair Encoding (BPE) Logic

Modern LLMs use **Sub-word Tokenization**. The most common algorithm is BPE. It iteratively merges the most frequent pair of adjacent characters.

Let's simulate one step of BPE.

In [2]:
from collections import Counter

# Sample corpus
corpus = "low low low lower newest"
# Split into words (pre-tokenization)
words = corpus.split()
# Represent words as list of characters
splits = [list(word) + ['</w>'] for word in words]

print("Initial splits:", splits)

def get_stats(vocab):
    pairs = Counter()
    for word in vocab:
        for i in range(len(word) - 1):
            pairs[word[i], word[i+1]] += 1
    return pairs

def merge_vocab(pair, v_in):
    v_out = []
    bigram = pair
    for word in v_in:
        w_out = []
        i = 0
        while i < len(word):
            # If we find the pair, merge it
            if i < len(word) - 1 and word[i] == bigram[0] and word[i+1] == bigram[1]:
                w_out.append(bigram[0] + bigram[1])
                i += 2
            else:
                w_out.append(word[i])
                i += 1
        v_out.append(w_out)
    return v_out

# Find most frequent pair
pairs = get_stats(splits)
best = max(pairs, key=pairs.get)
print(f"Most frequent pair: {best}")

# Merge it
splits = merge_vocab(best, splits)
print("After merging:", splits)

Initial splits: [['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', '</w>'], ['l', 'o', 'w', 'e', 'r', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>']]
Most frequent pair: ('l', 'o')
After merging: [['lo', 'w', '</w>'], ['lo', 'w', '</w>'], ['lo', 'w', '</w>'], ['lo', 'w', 'e', 'r', '</w>'], ['n', 'e', 'w', 'e', 's', 't', '</w>']]


This process repeats thousands of times to build a vocabulary of ~32k-100k tokens. This gives us a balance between sequence length and vocabulary size.

## 3. Using the Pre-trained Tokenizer

In our project, we use the tokenizer from `SmolLM-135M`. It has already been trained on a massive dataset.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
print(f"Vocab size: {tokenizer.vocab_size}")

text = "The quick brown fox jumps over the lazy dog."
tokens = tokenizer.encode(text)
print(f"Tokens: {tokens}")
print(f"Decoded: {[tokenizer.decode([t]) for t in tokens]}")

/Users/vukrosic/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocab size: 49152
Tokens: [504, 2365, 6354, 16438, 27003, 690, 260, 23790, 2767, 30]
Decoded: ['The', ' quick', ' brown', ' fox', ' jumps', ' over', ' the', ' lazy', ' dog', '.']
